In [1]:
import pandas as pd

df = pd.read_csv("../data/raw.csv", parse_dates=["InvoiceDate"])

print(df.shape)
print(df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
print(df.isna().sum())
print(df.dtypes)



(1067371, 8)
2009-12-01 07:45:00 to 2011-12-09 12:50:00
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64
Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object


In [2]:
df = df.dropna(subset=["Customer ID"]).copy()
df["Customer ID"] = df["Customer ID"].astype(int)

df["Invoice"] = df["Invoice"].astype(str)
df = df[~df["Invoice"].str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)].copy()

df["Revenue"] = df["Quantity"] * df["Price"]

print(df.shape)
print("Customers:", df["Customer ID"].nunique())
print("Total revenue:", round(df["Revenue"].sum(), 2))
print(df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())

(805549, 9)
Customers: 5878
Total revenue: 17743429.18
2009-12-01 07:45:00 to 2011-12-09 12:50:00


In [3]:
cutoff = pd.Timestamp("2011-09-09")
target_end = cutoff + pd.Timedelta(days=90)

history = df[df["InvoiceDate"] < cutoff].copy()
future = df[(df["InvoiceDate"] >= cutoff) & (df["InvoiceDate"] < target_end)].copy()

print("Cutoff:", cutoff.date(), "| Target window ends:", target_end.date())
print("History rows:", history.shape[0], "| customers:", history["Customer ID"].nunique())
print("Future rows:", future.shape[0], "| customers:", future["Customer ID"].nunique())
print("Last date in data:", df["InvoiceDate"].max().date())


Cutoff: 2011-09-09 | Target window ends: 2011-12-08
History rows: 642747 | customers: 5278
Future rows: 159652 | customers: 2877
Last date in data: 2011-12-09


In [4]:
features = history.groupby("Customer ID").agg(
    last_purchase=("InvoiceDate", "max"),
    first_purchase=("InvoiceDate", "min"),
    frequency=("Invoice", "nunique"),
    monetary=("Revenue", "sum"),
    total_items=("Quantity", "sum"),
    unique_products=("StockCode", "nunique"),
).reset_index()

features["recency_days"] = (cutoff - features["last_purchase"]).dt.days
features["tenure_days"] = (cutoff - features["first_purchase"]).dt.days
features["avg_order_value"] = features["monetary"] / features["frequency"]
features = features.drop(columns=["last_purchase", "first_purchase"])

print(features.shape)
print(features.head())
print(features.describe().round(1))

(5278, 8)
   Customer ID  frequency  monetary  total_items  unique_products  \
0        12346         12  77556.46        74285               27   
1        12347          6   4114.18         2418              107   
2        12348          4   1709.40         2497               25   
3        12349          3   2671.14          993               90   
4        12350          1    334.40          197               17   

   recency_days  tenure_days  avg_order_value  
0           233          633      6463.038333  
1            37          312       685.696667  
2           156          346       427.350000  
3           315          497       890.380000  
4           218          218       334.400000  
       Customer ID  frequency  monetary  total_items  unique_products  \
count       5278.0     5278.0    5278.0       5278.0           5278.0   
mean       15324.0        5.7    2702.2       1654.4             74.3   
std         1712.3       11.5   12536.3       8136.7            104.

In [5]:
future_spend = future.groupby("Customer ID")["Revenue"].sum().rename("future_spend")

customers = features.merge(future_spend, on="Customer ID", how="left")
customers["future_spend"] = customers["future_spend"].fillna(0)
customers["buys_again"] = (customers["future_spend"] > 0).astype(int)

customers.to_csv("../data/customers.csv", index=False)

print(customers.shape)
print(customers["buys_again"].value_counts())
print("Buy-again rate:", round(customers["buys_again"].mean(), 3))
print(customers["future_spend"].describe().round(1))

(5278, 10)
buys_again
0    2997
1    2281
Name: count, dtype: int64
Buy-again rate: 0.432
count      5278.0
mean        539.2
std        3199.8
min           0.0
25%           0.0
50%           0.0
75%         437.3
max      112721.0
Name: future_spend, dtype: float64
